# Chapitre 2 — PyTorch pour les réseaux de neurones

*De la convolution brute à un petit CNN entraîné sur CIFAR-10.*

Introduction **autonome** à PyTorch : aucune classe spécifique à GMIC. À la fin tu sauras :

- manipuler des tenseurs et l'auto-différentiation,
- écrire un `nn.Module` et faire passer des données dedans,
- comprendre ce que fait une `Conv2d` (filtre, stride, padding),
- entraîner un petit CNN sur CIFAR-10 avec une boucle forward / backward,
- comparer un petit CNN maison avec un ResNet-18 standard, entraîné à la bonne résolution.

C'est le préquel naturel des chapitres GMIC (ch4 architecture, ch5 fine-tuning). Le notebook
**télécharge CIFAR-10** (~170 Mo, automatique) et entraîne deux réseaux : prévois un **GPU**
(disponible dans le conteneur du cours) pour que ça tourne en ~1–2 min.

> Les organigrammes sont dessinés en **matplotlib** (sortie de cellule de code) plutôt qu'en
> Mermaid : ils s'affichent ainsi partout. Le helper `flowchart()` vient de `course_utils.py`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from course_utils import flowchart

plt.rcParams["figure.dpi"] = 110
torch.manual_seed(0)
np.random.seed(0)
torch.backends.cudnn.benchmark = True   # auto-tune cuDNN pour tailles d'entrée fixes

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch     : {torch.__version__}")
print(f"Device      : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU         : {torch.cuda.get_device_name(0)}")

# Partie I — Un tenseur, c'est un tableau avec des super-pouvoirs

Un **tenseur** PyTorch ressemble à un `np.array`, mais il sait :

1. tourner sur GPU (`.to("cuda")`) ;
2. enregistrer ses opérations pour calculer automatiquement les dérivées via `.backward()` —
   c'est l'**autograd**, le cœur de l'entraînement.

In [ ]:
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.tensor([[0., 1.], [1., 0.]])
print("a + b =\n", a + b)
print("a @ b =\n", a @ b)        # produit matriciel
print("a.shape =", a.shape, " | dtype =", a.dtype)

## Autograd en une image

On définit `y = x² + 3x + 2`, on demande à PyTorch de tracer les gradients, et on appelle
`.backward()`. La dérivée analytique est `dy/dx = 2x + 3`. En `x = 4` on attend donc `11`.

### Ce que PyTorch fabrique pendant le forward

Quand tu écris `y = x**2 + 3*x + 2`, PyTorch ne se contente pas de calculer un nombre : pour
chaque opération, il enregistre aussi *qui* étaient les entrées et *comment* remonter la
dérivée. Résultat : un **graphe de calcul** orienté, où `x` est la racine (une **feuille** :
tu l'as créé à la main) et `y` est la sortie (un **nœud intermédiaire**, fabriqué par PyTorch).
Le diagramme ci-dessous montre la passe avant (flèches pleines) et le résultat de la passe
arrière (chain rule).

In [ ]:
# Graphe de calcul de y = x² + 3x + 2 en x = 4 (forward plein -> backward résumé)
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis("off")

nodes = {
    "x":  (0.9, 3.0, "x = 4.0\n(feuille)",  "#e1f5fe"),
    "v1": (3.8, 4.4, "v1 = x² = 16",         "#fff3e0"),
    "v2": (3.8, 1.6, "v2 = 3·x = 12",        "#fff3e0"),
    "v3": (6.5, 3.0, "v3 = v1+v2 = 28",      "#fff3e0"),
    "y":  (9.0, 3.0, "y = v3+2 = 30",        "#e8f5e9"),
}
pos = {}
for name, (x, y, label, color) in nodes.items():
    ax.add_patch(FancyBboxPatch((x-0.85, y-0.34), 1.7, 0.68, boxstyle="round,pad=0.06",
                 facecolor=color, edgecolor="#555", linewidth=1.4))
    ax.text(x, y, label, ha="center", va="center", fontsize=9)
    pos[name] = (x, y)

def fwd(a, b):
    (x0, y0), (x1, y1) = pos[a], pos[b]
    ax.annotate("", xy=(x1-0.9, y1), xytext=(x0+0.9, y0),
                arrowprops=dict(arrowstyle="-|>", color="#1565c0", lw=1.8))

for a, b in [("x","v1"), ("x","v2"), ("v1","v3"), ("v2","v3"), ("v3","y")]:
    fwd(a, b)

ax.text(5, 0.4, "Backward (chain rule) :  dy/dx = dv1/dx + dv2/dx = 2x + 3 = 8 + 3 = 11",
        ha="center", fontsize=10, color="#c62828",
        bbox=dict(boxstyle="round,pad=0.4", fc="#fff5f5", ec="#c62828"))
ax.set_title("Graphe de calcul de y = x² + 3x + 2", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

**Pourquoi `x.grad` et pas `y.grad` ?** PyTorch ne stocke par défaut le gradient que sur les
**feuilles** (tenseurs créés par toi avec `requires_grad=True`). `y` est un intermédiaire : son
`.grad` reste `None`. On peut forcer avec `y.retain_grad()`, mais en pratique on n'a besoin que
des gradients des paramètres (les poids), qui sont justement des feuilles.

**Pourquoi un `.backward()` sur un scalaire ?** PyTorch ne sait dériver que des scalaires par
rapport à des tenseurs. Si `y` était un vecteur, il faudrait préciser par rapport à quelle
combinaison linéaire dériver (`grad_tensors=...`). D'où le `loss.backward()` toujours sur une
loss réduite à un seul nombre.

### Cycle de vie du graphe de calcul

Le graphe n'est pas gratuit : pour un gros modèle il peut peser plusieurs Go de RAM GPU.
PyTorch **le détruit par défaut après chaque `.backward()`** pour libérer la mémoire — un second
`.backward()` sur le même `y` plantera, sauf si `retain_graph=True`.

In [ ]:
flowchart([
    "1. Forward : y = x²+3x+2  -> alloue le graphe (variables temporaires C++)",
    "2. Backward : y.backward() -> calcule x.grad PUIS libère les temporaires",
    "3. Apres : la feuille x garde son .grad, le graphe est purge de la RAM",
], title="Cycle de vie du graphe de calcul")

In [ ]:
x = torch.tensor(4.0, requires_grad=True)
y = x ** 2 + 3 * x + 2
print("Avant backward :", x.grad)              # None — graphe créé, pas encore de gradient
y.backward()
print(f"y = {y.item()}")                        # 30.0 (valeur forward)
print(f"y.grad              : {y.grad}")        # None — y n'est pas une feuille
print(f"x.grad              : {x.grad.item()}") # 11.0 — gradient de la feuille
print(f"2x + 3 (analytique) : {2*4 + 3}")

### Les gradients s'**accumulent**

Piège classique : `.grad` ne s'écrase pas, il s'**additionne** à chaque `.backward()`. Sans
remise à zéro, on obtient la somme des dérivées. D'où l'obligation d'appeler
`optimizer.zero_grad()` (ou `x.grad.zero_()`) au début de chaque itération.

In [ ]:
x = torch.tensor(4.0, requires_grad=True)

y1 = x ** 2 + 3 * x + 2          # dy1/dx = 2x + 3 = 11
y1.backward()
print(f"après y1.backward : x.grad = {x.grad.item()}")   # 11

y2 = x ** 2                       # dy2/dx = 2x      = 8
y2.backward()
print(f"après y2.backward : x.grad = {x.grad.item()}")   # 19 = 11 + 8 (accumulation)

x.grad.zero_()                    # on remet le compteur à zéro
y3 = 5 * x
y3.backward()
print(f"après zero_() + y3 : x.grad = {x.grad.item()}")  # 5

# Partie II — Mon premier `nn.Module`

Un **module** PyTorch, c'est : (1) une classe qui hérite de `nn.Module` ; (2) un `__init__` qui
déclare les **couches à poids apprenables** ; (3) une méthode `forward(x)` qui décrit le calcul.
Construisons le plus petit réseau possible : une régression linéaire `y = w·x + b`.

In [ ]:
class MiniLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(in_features=1, out_features=1)

    def forward(self, x):
        return self.linear(x)

net = MiniLinear()
print(net)
print("Paramètres :", list(net.parameters()))

## Entraîner `MiniLinear` à apprendre `y = 2x + 1`

On assemble le module avec les trois autres briques : **des données**, **une loss** (à quel
point on se trompe), **un optimiseur** (qui met à jour les poids depuis le gradient). On fabrique
20 points bruités sur la droite `y = 2x + 1` et on demande au réseau de retrouver les coefficients.

In [ ]:
# 1. Données synthétiques : y_true = 2x + 1 + bruit gaussien
torch.manual_seed(0)
X = torch.linspace(-3, 3, 20).unsqueeze(1)        # shape (20, 1)
y_true = 2 * X + 1 + 0.3 * torch.randn_like(X)    # bruit d'écart-type 0.3

# 2. Le réseau, la loss et l'optimiseur
net = MiniLinear()
loss_fn = nn.MSELoss()                            # erreur quadratique moyenne
optimizer = torch.optim.SGD(net.parameters(), lr=0.05)

# 3. La boucle — les 5 lignes canoniques de PyTorch
losses = []
for step in range(200):
    y_pred = net(X)                               # (1) forward
    loss = loss_fn(y_pred, y_true)                # (2) comparer à la vérité
    optimizer.zero_grad()                         # (3) reset gradients
    loss.backward()                               # (4) rétro-propagation
    optimizer.step()                              # (5) mise à jour w et b
    losses.append(loss.item())
    if step % 40 == 0:
        print(f"step {step:3d}  loss = {loss.item():.4f}")

w = net.linear.weight.item(); b = net.linear.bias.item()
print(f"\nAppris : y ≈ {w:.3f} * x + {b:.3f}")
print(f"Vérité : y = 2.000 * x + 1.000")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(losses); ax[0].set_yscale("log")
ax[0].set_title("Loss (échelle log)"); ax[0].set_xlabel("itération")

ax[1].scatter(X.numpy(), y_true.numpy(), label="données bruitées", alpha=0.7)
x_line = torch.linspace(-3, 3, 100).unsqueeze(1)
with torch.no_grad():
    y_line = net(x_line)
ax[1].plot(x_line.numpy(), y_line.numpy(), "r-", lw=2, label=f"appris : y ≈ {w:.2f}x + {b:.2f}")
ax[1].plot(x_line.numpy(), (2 * x_line + 1).numpy(), "k--", alpha=0.5, label="vérité : y = 2x + 1")
ax[1].legend(); ax[1].set_title("Régression apprise")
plt.tight_layout(); plt.show()

Ce micro-entraînement est **la même boucle** qu'on utilisera pour CIFAR-10 (Partie VI), pour
ResNet-18 (Partie VIII) et pour le fine-tuning sur mammographies (ch3, ch5). Seuls changent
l'architecture, la loss et la source de données. Le squelette **forward → loss → zero_grad →
backward → step** est invariant.

# Partie III — `nn.Conv2d`, la brique des CNN

Pour classer une image, on ne traite pas chaque pixel comme une feature indépendante : il y en a
trop, et surtout l'information est **locale** (un œil, une roue, une tumeur). La convolution
résout ça. Une `Conv2d(in_channels=3, out_channels=8, kernel_size=3)` fait glisser **8 filtres**
3×3 sur l'entrée (3 canaux RGB) et produit **8 cartes d'activation**.

## Quatre filtres classiques sur une vraie image

On prend une image CIFAR-10 et on lui applique quatre **filtres codés à la main** :
**Sobel X** (bords verticaux), **Sobel Y** (bords horizontaux), **Laplacien** (tous les contours),
**Moyenneur 3×3** (flou). Idiomes PyTorch utilisés : les méthodes en `_` (`copy_`) sont
**in-place** ; `.view(1,1,3,3)` reshape sans copie vers `(out, in, H, W)` ; `with torch.no_grad():`
désactive le traçage (obligatoire pour écrire des poids in-place) ; `bias=False` évite un décalage
appris ; `.detach()` casse le lien avec le graphe.

In [ ]:
CIFAR_ROOT = os.path.expanduser("~/.cache/cifar10")
_demo_ds = datasets.CIFAR10(root=CIFAR_ROOT, train=True,
                            transform=transforms.ToTensor(), download=True)

# Une image de voiture (classe 1) pour avoir des bords nets
for _img, _lbl in _demo_ds:
    if _lbl == 1:
        img_rgb = _img            # (3, 32, 32) en [0, 1]
        break
img_gray = img_rgb.mean(0, keepdim=True).unsqueeze(0)   # (1, 1, 32, 32)

filters = {
    "Sobel X":   torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32),
    "Sobel Y":   torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32),
    "Laplacien": torch.tensor([[0, -1, 0], [-1, 4, -1], [0, -1, 0]], dtype=torch.float32),
    "Flou 3x3":  torch.ones(3, 3) / 9.0,
}

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
axes[0].imshow(img_rgb.permute(1, 2, 0)); axes[0].set_title("Entrée RGB"); axes[0].axis("off")
for ax, (name, k) in zip(axes[1:], filters.items()):
    conv = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
    with torch.no_grad():
        conv.weight.copy_(k.view(1, 1, 3, 3))
    feat = conv(img_gray)[0, 0].detach()
    ax.imshow(feat, cmap="gray"); ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()

**Clé pédagogique** : dans un vrai CNN, ces coefficients 3×3 ne sont pas codés à la main — ils
sont **appris** par descente de gradient. Ci-dessous, `nn.Conv2d(3, 8, 3)` crée 8 filtres
indépendants → 8 cartes d'activation. Avant entraînement, ils sont aléatoires (bruit structuré).

In [ ]:
torch.manual_seed(0)
conv = nn.Conv2d(3, 8, kernel_size=3, padding=1, bias=False)
with torch.no_grad():
    feat = conv(img_rgb.unsqueeze(0))    # (1, 8, 32, 32)

fig, axes = plt.subplots(1, 9, figsize=(14, 2))
axes[0].imshow(img_rgb.permute(1, 2, 0)); axes[0].set_title("Entrée", fontsize=9); axes[0].axis("off")
for i in range(8):
    axes[i + 1].imshow(feat[0, i].detach(), cmap="gray")
    axes[i + 1].set_title(f"Canal {i}", fontsize=9); axes[i + 1].axis("off")
plt.tight_layout(); plt.show()

## Effet des hyperparamètres

| Paramètre | Rôle | Effet sur la shape |
|:---|:---|:---|
| `kernel_size` | taille du filtre | plus grand = champ réceptif plus large |
| `stride` | pas de déplacement | stride 2 divise H et W par ~2 |
| `padding` | zéros ajoutés aux bords | préserve la shape d'entrée |
| `out_channels` | nombre de filtres | dimension du canal en sortie |

In [ ]:
x = torch.randn(1, 3, 32, 32)       # batch 1, 3 canaux, 32x32
for (k, s, p) in [(3, 1, 1), (3, 2, 1), (5, 1, 0), (7, 2, 3)]:
    c = nn.Conv2d(3, 8, kernel_size=k, stride=s, padding=p)
    out = c(x)
    print(f"kernel={k} stride={s} padding={p}  ->  in {tuple(x.shape)}  out {tuple(out.shape)}")

# Partie IV — CIFAR-10 et le pipeline de données

CIFAR-10 = 60 000 images 32×32 couleur, 10 classes. Avant de la donner au réseau, on passe par
trois briques que PyTorch impose pour **tout** entraînement :

| Brique | Rôle | Entrée | Sortie |
|:---|:---|:---|:---|
| `transforms` | **transformer** une image | `PIL.Image` / tenseur | tenseur prêt pour le réseau |
| `Dataset` | **exposer** les échantillons un par un | un index `i` | `(x_i, y_i)` |
| `DataLoader` | **batcher + paralléliser** | un `Dataset` | une séquence de batches |

### `transforms` — le pipeline par image
- `ToTensor()` : `PIL.Image` (uint8 0–255) → `float32` dans [0,1], convention **(C, H, W)**.
- `Normalize(mean, std)` : `(x - mean) / std`. Ici `0.5/0.5` mappe [0,1] → [-1,1]. Un réseau
  converge plus vite avec des entrées centrées/échelle ~1. Pour un modèle pré-entraîné ImageNet,
  on utilise les stats `mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]`.

### `Dataset` — l'interface minimale
Deux questions : `__len__` (combien ?) et `__getitem__(i)` (donne l'échantillon i). C'est tout.

In [ ]:
class MyDataset(Dataset):
    def __init__(self, xs, ys):
        self.xs, self.ys = xs, ys            # deux listes de même longueur
    def __len__(self):
        return len(self.xs)
    def __getitem__(self, i):
        return self.xs[i], self.ys[i]        # PyTorch ne demande rien de plus

tiny = MyDataset([torch.randn(3) for _ in range(4)], [0, 1, 1, 0])
print("len :", len(tiny))
print("item 2 :", tiny[2])

### `DataLoader` — le gestionnaire de batches
Il résout d'un coup : (1) **shuffle** des indices à chaque epoch ; (2) **batching** (empile
`batch_size` échantillons via `torch.stack`) ; (3) **parallélisation** avec `num_workers=N`
(sous-processus qui pré-calculent les batches — gain majeur quand `__getitem__` est lourd) ;
(4) **`pin_memory=True`** (transfert GPU en une DMA, gratuit sur CUDA). Ensuite on écrit juste
`for x, y in loader: ...`.

In [ ]:
CIFAR_ROOT = os.path.expanduser("~/.cache/cifar10")

transform = transforms.Compose([
    transforms.ToTensor(),                                  # [0,255] -> [0,1], (3,32,32)
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)), # -> [-1, 1]
])

train_full = datasets.CIFAR10(root=CIFAR_ROOT, train=True,  transform=transform, download=True)
test_full  = datasets.CIFAR10(root=CIFAR_ROOT, train=False, transform=transform, download=True)

# Sous-échantillons pour garder le notebook rapide
train_ds = Subset(train_full, range(10_000))
test_ds  = Subset(test_full,  range(2_000))

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=4,
                          pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=4,
                          pin_memory=True, persistent_workers=True)

classes = train_full.classes
print("Classes :", classes)
print(f"Train subset : {len(train_ds)} images  |  Test subset : {len(test_ds)} images")

imgs, labels = next(iter(train_loader))
print(f"Shape d'un batch : {imgs.shape}  (N, C, H, W)")
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    img = imgs[i].permute(1, 2, 0).numpy() * 0.5 + 0.5   # dé-normalisation pour affichage
    ax.imshow(img); ax.set_title(classes[labels[i]], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

Note la dé-normalisation `* 0.5 + 0.5` pour l'affichage : `matplotlib` veut des pixels dans
[0, 1] alors que `Normalize` nous a mis dans [-1, 1]. Le pipeline d'affichage est l'inverse du
pipeline d'entraînement.

# Partie V — Architecture d'un petit CNN

CNN simple à **deux** étages convolutifs + une tête fully-connected, sur le schéma canonique :
`Conv → BatchNorm → ReLU → MaxPool` (×N), puis `Flatten → Linear → ReLU → Linear(10)`.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        # Bloc 1 : 3->32 canaux, pool -> 16x16
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        # Bloc 2 : 32->64 canaux, pool -> 8x8
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        # Tête
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))   # (N, 32, 16, 16)
        x = self.pool(F.relu(self.bn2(self.conv2(x))))   # (N, 64,  8,  8)
        x = x.view(x.size(0), -1)                         # flatten (N, 4096)
        x = self.dropout(F.relu(self.fc1(x)))             # (N, 128)
        return self.fc2(x)                                # (N, 10) logits

model = SmallCNN().to(DEVICE)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nParamètres totaux : {n_params:,}  ({n_params/1e6:.2f} M)")

# Partie VI — La boucle d'entraînement

Les lignes magiques de **tout** entraînement PyTorch : `pred = model(x)` (forward) →
`loss = criterion(pred, y)` → `loss.backward()` → `optimizer.step()` → `optimizer.zero_grad()`.
Emballées dans deux boucles : sur les **époques** et sur les **batches**.

`criterion` n'est pas un mot-clé, juste une convention pour « la fonction de loss ».
`nn.CrossEntropyLoss` attend les **logits** (pas les probabilités) : elle applique
`LogSoftmax + NLLLoss` en interne — d'où le `forward` de `SmallCNN` qui renvoie `fc2(x)` sans
activation finale. Cas voisins : binaire → `BCEWithLogitsLoss` ; régression → `MSELoss`.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total, correct, running_loss = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
        correct += (pred.argmax(1) == y).sum().item()
        total += x.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total, correct, running_loss = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        running_loss += criterion(pred, y).item() * x.size(0)
        correct += (pred.argmax(1) == y).sum().item()
        total += x.size(0)
    return running_loss / total, correct / total


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
EPOCHS = 15
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    vl_loss, vl_acc = evaluate(model, test_loader, criterion, DEVICE)
    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss);  history["val_acc"].append(vl_acc)
    print(f"Epoch {epoch}/{EPOCHS}  "
          f"train loss={tr_loss:.3f} acc={tr_acc*100:.1f}%   "
          f"val loss={vl_loss:.3f} acc={vl_acc*100:.1f}%")
smallcnn_time = time.time() - t0
print(f"\nTemps total : {smallcnn_time:.1f} s")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot([a*100 for a in history["train_acc"]], label="train")
ax[1].plot([a*100 for a in history["val_acc"]],   label="val")
ax[1].set_title("Accuracy (%)"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

Si tout va bien : loss qui descend, accuracy qui monte. Avec 10 000 images, on tourne autour
de **60-70 %** sur CIFAR-10 — loin de l'état de l'art (~99 %) mais largement au-dessus du
**hasard (10 %)** pour un réseau entraîné en moins d'une minute.

> **Pourquoi « hasard = 10 % » ?** CIFAR-10 a **10 classes équilibrées** → un classifieur
> aléatoire tombe juste 1 fois sur 10. Pour N classes équilibrées le baseline est 1/N ; pour du
> binaire, 50 %.

# Partie VII — Prédictions visualisées

In [ ]:
model.eval()
imgs, labels = next(iter(test_loader))
with torch.no_grad():
    preds = model(imgs.to(DEVICE)).argmax(1).cpu()

fig, axes = plt.subplots(2, 6, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    img = imgs[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    ok = preds[i].item() == labels[i].item()
    color = "green" if ok else "red"
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"vrai: {classes[labels[i]]}\npréd: {classes[preds[i]]}",
                 fontsize=9, color=color)
plt.tight_layout(); plt.show()

# Partie VIII — ResNet-18 de `torchvision`, entraîné à sa résolution

On instancie un **ResNet-18** (He et al. 2015, 18 couches) sans poids pré-entraînés.

## Pourquoi la résolution change tout

ResNet-18 est conçu pour ImageNet (224×224). Avec des entrées 32×32, son `conv1` (7×7, stride 2)
+ `MaxPool` réduit l'image à 8×8, puis les blocs la compressent jusqu'à **1×1** avant le
classifieur — toute l'info spatiale est perdue. On **upsample CIFAR-10 à 96×96** (bicubique).

| Étage ResNet-18 | images 32×32 | **images 96×96** |
|:---|:---:|:---:|
| Après `conv1` + `maxpool` | 8×8 | **24×24** |
| Après `layer2` (stride 2) | 4×4 | 12×12 |
| Après `layer3` (stride 2) | 2×2 | 6×6 |
| Après `layer4` (stride 2) | **1×1** | **3×3** |

À 96×96 le Global Avg Pool final travaille sur 9 positions au lieu d'une seule.

In [ ]:
rn18_transform = transforms.Compose([
    transforms.Resize((96, 96), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

rn18_train_ds = Subset(datasets.CIFAR10(root=CIFAR_ROOT, train=True,
                                        transform=rn18_transform, download=True), range(10_000))
rn18_test_ds  = Subset(datasets.CIFAR10(root=CIFAR_ROOT, train=False,
                                        transform=rn18_transform, download=True), range(2_000))

rn18_train_loader = DataLoader(rn18_train_ds, batch_size=64,  shuffle=True,  num_workers=4,
                               pin_memory=True, persistent_workers=True)
rn18_test_loader  = DataLoader(rn18_test_ds,  batch_size=128, shuffle=False, num_workers=4,
                               pin_memory=True, persistent_workers=True)
print(f"Taille des batches : {next(iter(rn18_train_loader))[0].shape}")

In [ ]:
from torchvision.models import resnet18

resnet18_model = resnet18(weights=None, num_classes=10).to(DEVICE)
n_rn18 = sum(p.numel() for p in resnet18_model.parameters())
print(f"ResNet-18 : {n_rn18:,} paramètres  ({n_rn18/1e6:.2f} M)")
print(resnet18_model.conv1)

In [ ]:
rn18_optimizer = torch.optim.Adam(resnet18_model.parameters(), lr=1e-3)
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(resnet18_model, rn18_train_loader,
                                      rn18_optimizer, criterion, DEVICE)
    vl_loss, vl_acc = evaluate(resnet18_model, rn18_test_loader, criterion, DEVICE)
    print(f"Epoch {epoch}/{EPOCHS}  train acc={tr_acc*100:.1f}%   val acc={vl_acc*100:.1f}%")
resnet18_time = time.time() - t0
print(f"\nTemps total : {resnet18_time:.1f} s")

# Partie IX — Tableau comparatif

Accuracy top-1 et **AUC ROC macro** sur le test set. SmallCNN jugé sur ses 32×32, ResNet-18 sur
ses 96×96 — chaque modèle dans les conditions où il a été entraîné.

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score

@torch.no_grad()
def compute_metrics(net, loader, device):
    net.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        probs = F.softmax(net(x.to(device)), dim=1).cpu().numpy()
        all_probs.append(probs); all_labels.append(y.numpy())
    probs = np.concatenate(all_probs); labels = np.concatenate(all_labels)
    acc = accuracy_score(labels, probs.argmax(1))
    auc = roc_auc_score(labels, probs, multi_class="ovr", average="macro")
    return acc, auc

rows = []
for name, net, loader, t in [
    ("SmallCNN",  model,          test_loader,      smallcnn_time),
    ("ResNet-18", resnet18_model, rn18_test_loader, resnet18_time),
]:
    acc, auc = compute_metrics(net, loader, DEVICE)
    n = sum(p.numel() for p in net.parameters())
    rows.append((name, n, t, acc, auc))

print(f"{'Modèle':<13}{'Paramètres':>14}{'Temps (s)':>12}{'Accuracy':>12}{'AUC macro':>12}")
print("-" * 63)
for name, n, t, acc, auc in rows:
    print(f"{name:<13}{n:>14,}{t:>12.1f}{acc*100:>11.1f}%{auc:>12.3f}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11, 3.5))
names  = [r[0] for r in rows]
colors = ["#4C78A8", "#54A24B"]

ax[0].bar(names, [r[1]/1e6 for r in rows], color=colors); ax[0].set_title("Paramètres (M)")
ax[1].bar(names, [r[2] for r in rows], color=colors);     ax[1].set_title("Temps d'entraînement (s)")
ax[2].bar(names, [r[3]*100 for r in rows], color=colors, label="Accuracy %")
ax[2].plot(names, [r[4]*100 for r in rows], "ko--", label="AUC macro × 100")
ax[2].set_title("Qualité (acc. / AUC)"); ax[2].legend(fontsize=8)
for a in ax:
    a.tick_params(axis="x", rotation=10)
plt.tight_layout(); plt.show()

## Interprétation

- **Paramètres** : ResNet-18 (≈11 M) est ~22× plus gros — plus de capacité mais plus de risque
  d'overfitting avec 10 000 images.
- **Temps** : ResNet-18 est plus lent (plus de poids, images plus grandes, batches plus petits).
- **Qualité** : avec 96×96 et 15 epochs, ResNet-18 devrait dépasser SmallCNN. Si l'écart reste
  faible → il faudrait plus de données ou de l'augmentation (random crop, flip).

**AUC ROC macro** (un-contre-tous moyenné) mesure la qualité du *ranking* des probabilités,
indépendamment du seuil. 1.0 = parfait, 0.5 = hasard. Plus informative que l'accuracy quand les
classes sont déséquilibrées — cas typique en médical (une tumeur = quelques % des images).

## Les primitives à retenir

| Concept | Classe/fonction | Rôle |
|:---|:---|:---|
| Tenseur | `torch.tensor`, `.to(device)` | donnée de base |
| Dérivation | `.backward()`, `.grad` | autograd |
| Couche | `nn.Linear`, `nn.Conv2d`, `nn.BatchNorm2d` | poids apprenables |
| Module | `nn.Module` + `forward(x)` | composition |
| Non-linéarité | `F.relu`, `torch.sigmoid` | sans poids |
| Loss | `nn.CrossEntropyLoss`, `nn.MSELoss` | mesure d'écart |
| Optimiseur | `torch.optim.Adam`, `torch.optim.SGD` | descente de gradient |
| Données | `Dataset`, `DataLoader` | pipeline de batches |
| Régularisation | `nn.Dropout`, `nn.BatchNorm2d` | stabilité + généralisation |

Avec ces briques tu peux lire **n'importe quel** code PyTorch moderne, y compris GMIC (ch4).

Chapitre suivant → `02.5_preprocessing.ipynb`.

## 🧪 Smoke test

Vérifie que le code clé de ce chapitre tourne **sans télécharger les données ni lancer le preprocessing complet** (entrées synthétiques).

In [ ]:
# 🧪 Smoke test — valide la mécanique d'entraînement SANS télécharger CIFAR-10.
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)
# Données synthétiques : 2 classes séparables (64 images 1x8x8)
_X = torch.randn(64, 1, 8, 8); _y = (_X.mean((1, 2, 3)) > 0).long()
_loader = DataLoader(TensorDataset(_X, _y), batch_size=16, shuffle=True)

_net = nn.Sequential(nn.Flatten(), nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 2))
_opt = torch.optim.Adam(_net.parameters(), lr=1e-2)
_crit = nn.CrossEntropyLoss()
_first = _last = None
for _ep in range(20):
    for xb, yb in _loader:
        _opt.zero_grad(); _loss = _crit(_net(xb), yb); _loss.backward(); _opt.step()
    if _first is None:
        _first = _loss.item()
    _last = _loss.item()
assert _last < _first, f'la loss ne baisse pas ({_first:.3f} -> {_last:.3f})'

# Autograd : dy/dx de y = x²+3x+2 en x=4 vaut 2*4+3 = 11
_x = torch.tensor(4.0, requires_grad=True); (_x**2 + 3*_x + 2).backward()
assert abs(_x.grad.item() - 11.0) < 1e-6, _x.grad.item()
print(f'✅ Boucle train (loss {_first:.3f} -> {_last:.3f}) + autograd (x.grad=11) OK, sans CIFAR.')